# Docling Chunking

**Fuentes:**

- [Chunking Strategies in Docling](https://github.com/docling-project/docling-graph/blob/main/docs/fundamentals/extraction-process/chunking-strategies.md)
- [Docling Lab 2: Enhanced Document-Centric Chunking](https://github.com/docling-project/docling-workshops/blob/main/workshops/2025_09_24/docling_lab_2.ipynb)

Una vez que hemos convertido los documentos con Docling, ahora necesitamos fragmentar inteligentemente esos documentos para que la recuperaci?n en el RAG sea eficiente. Este es un paso crucial.

La generaci?n aumentada por recuperaci?n (RAG) se ha convertido en el patr?n dominante para crear aplicaciones de IA que requieren acceso a datos privados o recientes. A diferencia de los modelos de lenguaje puro, que se basan ?nicamente en datos de entrenamiento, los sistemas RAG recuperan din?micamente informaci?n relevante para generar respuestas precisas y fundamentadas.

Pensemos en RAG como si otorg?semos a un asistente de IA acceso a una biblioteca perfectamente organizada. La calidad de la organizaci?n (fragmentaci?n) determina directamente la capacidad del asistente para encontrar y utilizar la informaci?n.

El proceso RAG completo consta de seis pasos interconectados:

1. **Procesamiento de documentos**: Convertir documentos sin procesar (PDF, Word, etc.) a formatos estructurados.
2. **Fragmentaci?n**: Dividir los documentos en fragmentos sem?nticamente significativos.
3. **Vectorizaci?n**: Convertir fragmentos de texto en *embeddings* o vectores num?ricos.
4. **Indexaci?n**: Almacenar estos vectores en una base de datos especializada para la b?squeda por similitud.
5. **Recuperaci?n**: Encontrar los fragmentos m?s relevantes para una consulta dada.
6. **Generaci?n**: Usar el contexto recuperado para generar respuestas precisas y fundamentadas.

## ?Por qu? es crucial la fragmentaci?n (_chunking_)?

La fragmentaci?n es posiblemente el paso m?s importante en el flujo de trabajo de RAG. He aqu? por qu?:

### 1. Restricciones de la ventana de contexto

Los modelos de embeddings modernos tienen l?mites estrictos de tokens:

- IBM Granite embeddings: 512 tokens
- Cohere embed-v3: 512 tokens
- OpenAI's text-embedding-3-small: 8.191 tokens

Los fragmentos (_chunks_) deben ajustarse a estos l?mites, conservando al mismo tiempo el significado.

### 2. Compensaci?n entre precisi?n y sensibilidad

- **Fragmentos peque?os**: Alta precisi?n (coincidencias exactas), pero pueden perder contexto
- **Fragmentos grandes**: Mejor contexto, pero menor precisi?n (contenido irrelevante)

Encontrar el punto ?ptimo es crucial para el rendimiento del sistema.

### 3. Coherencia sem?ntica

Los fragmentos bien dise?ados mantienen la unidad tem?tica. Un fragmento sobre "ingresos trimestrales" no deber?a cambiar repentinamente a "beneficios para empleados" a mitad de camino.

### 4. Coste y rendimiento

- Fragmentos m?s peque?os = M?s embeddings = Mayores costes
- Fragmentos m?s grandes = Menos embeddings, pero potencialmente peor recuperaci?n

#### 5. Experiencia del usuario

La calidad de los fragmentos recuperados impacta directamente en la relevancia y precisi?n de las respuestas generadas. Una fragmentaci?n deficiente provoca alucinaciones, informaci?n ausente o respuestas irrelevantes.

## El desaf?o del *chuncking*: un ejemplo pr?ctico

Ilustremos por qu? la fragmentaci?n simple falla y la fragmentaci?n inteligente tiene ?xito:

**Ejemplo de fragmentaci?n incorrecta (divisi?n de tama?o fijo):**

```
Fragmento 1: "Los ingresos de la empresa aumentaron un 25 % en el tercer trimestre"
Fragmento 2: "de 2024 en comparaci?n con el tercer trimestre de 2023. Este crecimiento fue impulsado por..."
```

**Problemas**:

- El contexto cr?tico (?qu? a?o?) est? dividido en fragmentos
- Una b?squeda de "crecimiento de ingresos en 2024" podr?a pasar por alto el Fragmento 1 por completo
- El modelo carece de informaci?n completa para responder con precisi?n

**Ejemplo de fragmentaci?n correcta (consciente de la sem?ntica):**

```
Fragmento 1: "Rendimiento Financiero del Tercer Trimestre de 2024: Los ingresos de la empresa aumentaron un 25 % en el tercer trimestre de 2024 en comparaci?n con el tercer trimestre de 2023, alcanzando los 1200 millones de d?lares en ventas totales".
Fragmento 2: "Impulsores del crecimiento: Este crecimiento excepcional se debi? al s?lido desempe?o del segmento empresarial, donde los servicios en la nube contribuyeron con el 60 % del aumento..."
```

**Beneficios**:

- Ideas completas e independientes
- L?mites tem?ticos claros
- Contexto suficiente para una recuperaci?n precisa
- Se conservan los saltos de secci?n naturales

**?Qu? hace que un fragmento sea bueno?**

Antes de profundizar en la implementaci?n, definamos las caracter?sticas de los chuncks efectivos:

1. **Aut?nomos**: Se pueden entender sin contexto externo.
2. **Coherentes tem?ticamente**: Se centran en un solo tema o concepto.
3. **De tama?o adecuado**: Se ajustan a los l?mites del modelo con margen de maniobra.
4. **Contextualmente ricos**: Incluyen los metadatos necesarios (fuente, secci?n, p?gina).
5. **Sem?nticamente completos**: No se interrumpen a mitad de frase ni de pensamiento.

## Setup

In [ ]:
import sys
import torch

print(f"{sys.version=}")
print(f"{torch.__version__=}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Markdown
from typing import Iterator
from docling_core.transforms.chunker.tokenizer.base import BaseTokenizer
from docling.chunking import BaseChunker, BaseChunk
from docling.datamodel.document import DoclingDocument
import rich
import tiktoken
import tiktoken.model as tk_model
from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer
from docling_core.transforms.chunker.hierarchical_chunker import HierarchicalChunker
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker

In [ ]:
PATH_DATA = Path("..") / "data"
PATH_INPUT = PATH_DATA / "interim" / "docling" / "raw"
PATH_OUTPUT = PATH_DATA / "processed"

PATH_OUTPUT.mkdir(exist_ok=True, parents=True)

## Cargar documento a fragmentar

In [ ]:
# Archivos a procesar
pdf_files = sorted(PATH_INPUT.glob("*.json"))
pdf_files

In [ ]:
# Seleccionamos un documento para fragmentar
FILE_NAME = pdf_files[0].stem
FILE_NAME

In [ ]:
# Importamos su JSON y reconstruimos el DoclingDocument
json_path = PATH_INPUT / f"{FILE_NAME}.json"
doc = DoclingDocument.load_from_json(json_path)

type(doc)

Antes de poder fragmentar, necesitamos comprender c?mo Docling preserva la estructura del documento. Esta comprensi?n estructural es lo que distingue la fragmentaci?n ingenua de texto de la fragmentaci?n sem?ntica.

Docling no solo extrae texto, sino que preserva la estructura l?gica del documento. Esto incluye:

- **Encabezados y subencabezados**: L?mites naturales de los fragmentos
- **P?rrafos**: Unidades sem?nticas de pensamiento
- **Listas y tablas**: Requieren un manejo especial
- **Subt?tulos y referencias**: Contexto importante

Esta informaci?n estructural es fundamental para una fragmentaci?n inteligente.

## Visualizaci?n de los chunks

Antes de profundizar en las estrategias de fragmentaci?n, vamos a construir una herramienta de visualizaci?n, ya que poder "ver" los fragmentos es fundamental para comprender y depurar.

### Building a Comprehensive Chunk Analyzer

In [ ]:
def visualize_chunks(
    chunks: list[BaseChunk], *, chunker: BaseChunker, tokenizer: BaseTokenizer, title="Document Chunks"
):
    """Visualize chunk sizes and distribution in tokens."""
    # Extract token counts for each chunk
    token_counts = [tokenizer.count_tokens(chunker.contextualize(chunk=chunk)) for chunk in chunks]

    # Create histogram with all annotations in one go
    plt.figure(figsize=(10, 6))

    # Create the histogram
    plt.hist(token_counts, bins=20, alpha=0.7, color="skyblue")

    # Add statistics line and annotations
    avg_tokens = np.mean(token_counts)
    plt.axvline(avg_tokens, color="red", linestyle="--", label=f"Average: {avg_tokens:.1f}")

    # Add labels and formatting
    plt.title(title)
    plt.xlabel("Chunk Size (tokens)")
    plt.ylabel("Frequency")
    plt.grid(axis="y", alpha=0.75)
    plt.legend()

    # Show the complete plot
    plt.show()

    # Print comprehensive statistics
    print("Chunk Analysis Results:")
    print(f"Total chunks: {len(token_counts)}")
    print(f"Average chunk size: {np.mean(token_counts):.1f} tokens")
    print(f"Minimum chunk size: {min(token_counts)} tokens")
    print(f"Maximum chunk size: {max(token_counts)} tokens")
    print(f"Standard deviation: {np.std(token_counts):.1f} tokens")

    # Quality indicators
    if max(token_counts) > 512:
        print("Warning: Some chunks exceed 512 tokens - consider reducing chunk size")
    if np.std(token_counts) > 100:
        print("Warning: High variance in chunk sizes - retrieval consistency may suffer")

    # Also show character length for reference
    char_lengths = [len(chunk.text) for chunk in chunks]
    print(f"\nReference - Average character length: {np.mean(char_lengths):.1f} characters")

## Docling Chunking: conceptos b?sicos

Antes de explorar estrategias espec?ficas, comprendamos la arquitectura de fragmentaci?n de Docling. Este conocimiento nos ayudar? a ampliar o personalizar la fragmentaci?n seg?n nuestras necesidades.

### La interfaz `BaseChunker`

Docling define una interfaz `BaseChunker` que todos los fragmentadores deben implementar. Esta interfaz incluye dos m?todos principales:

1. `chunk(dl_doc)`: Devuelve un iterador de fragmentos para el documento proporcionado.
2. `serialize(chunk)`: Devuelve la representaci?n serializada de un fragmento, que se utiliza normalmente para los _embeddings_.

Examinemos esta interfaz:

In [ ]:
# Example of BaseChunker interface structure
class SimpleChunker(BaseChunker):
    """A simple example chunker implementing the BaseChunker interface."""

    def chunk(self, dl_doc: DoclingDocument, **kwargs) -> Iterator[BaseChunk]:
        """Return chunks for the provided document."""
        # Simple implementation: one chunk per page
        for i, page in enumerate(dl_doc.pages):
            text = " ".join([item.text for item in page.items if hasattr(item, "text")])
            metadata = {"page": i, "source": dl_doc.name}
            yield BaseChunk(text=text, metadata=metadata)

    def serialize(self, chunk: BaseChunk) -> str:
        """Serialize a chunk for embedding."""
        # Simple serialization: just return the text
        return chunk.text

### Estrategias de fragmentaci?n

Ahora exploremos las estrategias de fragmentaci?n integradas de Docling. Cada una tiene sus ventajas y casos de uso ideales.

#### Estrategia 1: `HierarchicalChunker`

`HierarchicalChunker` preserva la organizaci?n/estructura natural de los documentos siguiendo secciones, subsecciones y p?rrafos. Es ideal para documentos estructurados como informes, manuales y art?culos acad?micos.

**Cu?ndo usar `HierarchicalChunker`:**

- Documentos estructurados (informes, manuales, art?culos acad?micos)
- Cuando es importante preservar la jerarqu?a del documento
- Documentos con saltos de secci?n claros

**Evitar:** Registros de chat, publicaciones en redes sociales y texto no estructurado

Para saber qu? modelos de `tiktoken` tenemos disponibles.

In [ ]:
# Mapeos exactos de modelo -> encoding
# print(tk_model.MODEL_TO_ENCODING)

# Prefijos soportados (por ejemplo "gpt-4o-" -> encoding)
print(tk_model.MODEL_PREFIX_TO_ENCODING)

In [ ]:
tokenizer = OpenAITokenizer(
    tokenizer=tiktoken.encoding_for_model("gpt-4.1-"),
    max_tokens=512,
)

`max_tokens` es el tama?o m?ximo de los chunks, tiene que ser m?s menor o igual que el m?ximo de entrada del modelo de *embeddings*.

In [ ]:
# Create a HierarchicalChunker
hierarchical_chunker = HierarchicalChunker()

# Generate chunks
hierarchical_chunks = list(hierarchical_chunker.chunk(doc))

# Visualize the chunks
print(f"Generated {len(hierarchical_chunks)} chunks with HierarchicalChunker")
visualize_chunks(
    chunks=hierarchical_chunks,
    title="HierarchicalChunker Chunks",
    chunker=hierarchical_chunker,
    tokenizer=tokenizer,
)

# Examine chunk structure
sample_chunk = hierarchical_chunks[2]
print("\nSample Chunk Analysis:")
print(f"  - Text: '{sample_chunk.text}'")
print(f"  - Chunk type: {type(sample_chunk).__name__}")

# Print available metadata
print("\nDocument metadata:")
rich.print(sample_chunk.meta)

In [ ]:
rich.print(hierarchical_chunks[4].text)

In [ ]:
rich.print(hierarchical_chunker.contextualize(chunk=hierarchical_chunks[4]))

In [ ]:
rich.print(hierarchical_chunks[48].text)

In [ ]:
rich.print(hierarchical_chunker.contextualize(chunk=hierarchical_chunks[48]))

#### Estrategia 2: HybridChunker

Esta estrategia ofrece un wquilibrio entre estructura y tama?o. As?, `HybridChunker` combina lo mejor de ambos mundos: comienza con la fragmentaci?n jer?rquica, pero luego aplica divisiones y fusiones que tienen en cuenta el tama?o. Esto garantiza que los fragmentos no sean ni demasiado grandes ni demasiado peque?os.

**Cu?ndo usar HybridChunker:**

- La mayor?a de los sistemas RAG de producci?n (se recomienda como opci?n predeterminada)
- Cuando se necesitan tama?os de fragmento consistentes
- Tipos de documentos mixtos en el mismo sistema
- Cuando el modelo de *embeddings* tiene l?mites estrictos de tokens

In [ ]:
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer,
    ChunkingSerializerProvider,
)
from docling_core.transforms.serializer.markdown import MarkdownTableSerializer


class MDTableSerializerProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(
            doc=doc,
            table_serializer=MarkdownTableSerializer(),  # configuring a different table serializer
        )

In [ ]:
# Create a HybridChunker with default settings
hybrid_chunker = HybridChunker(
    tokenizer=tokenizer,
    serializer_provider=MDTableSerializerProvider(),
)

# Generate chunks
hybrid_chunks = list(hybrid_chunker.chunk(doc))

# Analyze the results
print("HybridChunker Results:")
print(f"Generated {len(hybrid_chunks)} chunks")

visualize_chunks(
    chunks=hybrid_chunks,
    title="HybridChunker: Structure + Size Aware",
    chunker=hybrid_chunker,
    tokenizer=tokenizer,
)

# Compare with HierarchicalChunker
print("\nStrategy Comparison:")
print(f"HierarchicalChunker: {len(hierarchical_chunks)} chunks")
print(f"HybridChunker: {len(hybrid_chunks)} chunks")
print(f"Reduction: {((len(hierarchical_chunks) - len(hybrid_chunks)) / len(hierarchical_chunks) * 100):.1f}%")

# Examine a sample chunk with context
sample_hybrid_chunk = hybrid_chunks[0]
print("\nSample HybridChunker Chunk:")
print(f"Text: '{sample_hybrid_chunk.text}'")

# Print available metadata
print("\nDocument metadata:")
rich.print(sample_hybrid_chunk.meta)

In [ ]:
rich.print(hybrid_chunks[0])

In [ ]:
rich.print(hybrid_chunks[24].text)

In [ ]:
rich.print(hybrid_chunker.contextualize(chunk=hybrid_chunks[24]))

In [ ]:
len(hybrid_chunks)

In [ ]:
from docling_core.transforms.chunker.base import BaseChunk
from docling_core.transforms.chunker.hierarchical_chunker import DocChunk
from docling_core.types.doc.labels import DocItemLabel

In [ ]:
table_chunks = []
for i, chunk in enumerate(hybrid_chunks):
    doc_chunk = DocChunk.model_validate(chunk)
    for it in doc_chunk.meta.doc_items:
        if it.label == DocItemLabel.TABLE:
            table_chunks.append(chunk)

In [ ]:
for item in table_chunks[0].meta.doc_items:
    print(f"Item label: {item.self_ref}, text: '{item.label}'")

In [ ]:
table_chunks[1].text

In [ ]:
display(Markdown(table_chunks[1].text))

## Configuraci?n avanzada y ajuste fino

Ahora exploremos las configuraciones avanzadas para optimizar la fragmentaci?n seg?n necesidades espec?ficas. La clave est? en comprender c?mo los diferentes par?metros afectan a los resultados.

### Comprensi?n del impacto del tama?o de fragmento

Los diferentes tama?os de fragmento tienen diferentes prop?sitos. Veamos c?mo configurar HybridChunker para necesidades espec?ficas:

In [ ]:
max_tokens = 128  # Smaller chunks for fine-grained retrieval

adv_tokenizer = OpenAITokenizer(
    tokenizer=tiktoken.encoding_for_model("gpt-4.1-"),
    max_tokens=max_tokens,
)

adv_chunker = HybridChunker(
    tokenizer=adv_tokenizer,
)

adv_chunks = list(adv_chunker.chunk(doc))

print(f"Advanced HybridChunker results ({max_tokens} token limit):")
print(f"Generated {len(adv_chunks)} chunks")

visualize_chunks(
    chunks=adv_chunks,
    title=f"HybridChunker with {max_tokens} token limit",
    chunker=adv_chunker,
    tokenizer=adv_tokenizer,
)

Tengamos en cuenta que el tama?o m?ximo de fragmento ahora es 128, como era de esperar.

### Contextualizaci?n

Otra de las funciones avanzadas de Docling es la contextualizaci?n: a?adir informaci?n relevante del entorno a los fragmentos para una mejor recuperaci?n. Ve?mosla en acci?n:

In [ ]:
for i, chunk in enumerate(adv_chunks[:5]):
    tokens_text = adv_tokenizer.count_tokens(chunk.text)
    contextualized = adv_chunker.contextualize(chunk)
    tokens_contextualized = adv_tokenizer.count_tokens(contextualized)

    print(f"Chunk {i}:")
    print(f"Original text ({tokens_text} tokens): {chunk.text}")
    print(f"Contextualized ({tokens_contextualized} tokens): {contextualized}")
    print(f"Context added: {tokens_contextualized - tokens_text} tokens\n")